In [2]:
import pandas as pd
from pathlib import Path

folder = Path('/home/edu/Unesp/IC/benchmark/results_oficial')

dfs = []

for file_path in folder.glob('*.csv'):
    df = pd.read_csv(file_path).tail(10)
    df = df[df['duration'] == 10000]

    file_name = file_path.name

    if 'ADJACENCY_LIST' in file_name:
        df["Implementation"] = 'ADJACENCY_LIST'

    if 'ADJACENCY_MATRIX' in file_name:
        df["Implementation"] = 'ADJACENCY_MATRIX'

    if 'CSR' in file_name:
        df["Implementation"] = 'CSR'

    if 'mostly_read' in file_name:
        df["Workload"] = 'mostly_read'

    if 'mostly_write' in file_name:
        df["Workload"] = 'mostly_write'

    if 'read_only' in file_name:
        df["Workload"] = 'read_only'


    dfs.append(df)

df = pd.concat(dfs,  ignore_index=True)

df.to_csv('results_oficial/complete_benchmark.csv')



In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

df = pd.read_csv('./results_oficial/complete_benchmark.csv')
df = df[df['duration'] == 10000]

df['throughput_total'] = (
    df['v_insert'] + df['v_del'] + 
    df['e_insert'] + df['e_del'] + 
    df['e_searches']
)
df['throughput_update'] = (
    df['v_insert'] + df['v_del'] + 
    df['e_insert'] + df['e_del']
)
df['throughput_read'] = df['e_searches']

throughputs = ['throughput_total', 'throughput_update', 'throughput_read']
df['throughput_total'] = df['throughput_total'] / 10
df['throughput_update'] = df['throughput_update'] / 10
df['throughput_read'] = df['throughput_read'] / 10



import pandas as pd
import numpy as np
import scipy.stats as stats

# Your setup
group_cols = ['size', 'duration', 'density', 'Implementation', 'Workload']
throughputs = ['throughput_total', 'throughput_update', 'throughput_read']# Replace with your actual metric names

# 1. Create a list to hold the summary data for each group
summary_data = []

# 2. Iterate through groups and calculate metrics
for group_name, group_df in df.groupby(group_cols):
    
    # Store the group keys (e.g., size, duration, etc.) so we can merge later
    group_stats = dict(zip(group_cols, group_name))
    n = len(group_df)
    
    # 3. Calculate mean and CI for each of the 3 metrics
    for t in throughputs:
        mean = group_df[t].mean()
        
        if n < 2:
            # Not enough data for a confidence interval
            ci_lower, ci_upper = np.nan, np.nan 
        else:
            se = stats.sem(group_df[t])
            ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
            
        # Add the new calculated values to our dictionary
        group_stats[f'{t}_mean'] = mean
        group_stats[f'{t}_ci_lower'] = ci_lower
        group_stats[f'{t}_ci_upper'] = ci_upper
        
    summary_data.append(group_stats)

# 4. Convert the calculated stats into a new DataFrame
summary_df = pd.DataFrame(summary_data)

# 5. Merge the new columns back into the original DataFrame!
#'how="left"' ensures we keep all original rows and just attach the new data
df_final = df.merge(summary_df, on=group_cols, how='left')

print(df_final.head())
summary_df.to_csv('results_oficial/confidence_interval.csv')
        

   Unnamed: 0.1  size  duration  density  v_updates  e_updates  e_searchs  \
0             0  1000     10000     0.75          5         20        100   
1             1  1000     10000     0.75          5         20        100   
2             2  1000     10000     0.75          5         20        100   
3             3  1000     10000     0.75          5         20        100   
4             4  1000     10000     0.75          5         20        100   

   mode  factor  v_insert  ...  throughput_read  throughput_total_mean  \
0     0       1   5109219  ...       16343142.1            20474262.68   
1     0       1   5122317  ...       16385378.4            20474262.68   
2     0       1   5123088  ...       16393119.4            20474262.68   
3     0       1   5124123  ...       16400075.5            20474262.68   
4     0       1   5125074  ...       16397458.9            20474262.68   

   throughput_total_ci_lower  throughput_total_ci_upper  \
0               2.045007e+07     

/home/edu/Unesp/IC/ic/lib/python3.14/site-packages/scipy/stats/_distn_infrastructure.py:2337: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/home/edu/Unesp/IC/ic/lib/python3.14/site-packages/scipy/stats/_distn_infrastructure.py:2338: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
import os

def build_dashboard_sheet(wb, sheet_name, df, metrics, row_col, group_col, x_col, temp_images):
    """Helper function to build a specific layout on a new Excel sheet."""
    ws = wb.create_sheet(title=sheet_name)
    ws.sheet_view.showGridLines = False
    
    # Title
    ws.cell(row=1, column=2, value=f"{sheet_name} Dashboard").font = Font(size=22, bold=True, color="1F4E79")
    ws.cell(row=2, column=2, value=f"X: {x_col.capitalize()} | Y: Mean Throughput (log ops/sec)").font = Font(size=12, italic=True)
    
    current_row = 4
    workload_order = ['read_only', 'mostly_read', 'mostly_write']
    
    # Helper to sort values dynamically based on column type
    def get_sorted_vals(col):
        vals = df[col].unique()
        if col == 'Workload':
            return [w for w in workload_order if w in vals]
        return sorted(vals)
        
    row_vals = get_sorted_vals(row_col)
    group_vals = get_sorted_vals(group_col)
    
    for r_val in row_vals:
        # Row Header
        ws.cell(row=current_row, column=2, value=f"{row_col.capitalize()}: {r_val}").font = Font(size=16, bold=True, color="FFFFFF")
        ws.cell(row=current_row, column=2).fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
        current_row += 2
        
        for g_val in group_vals:
            # Group Header
            ws.cell(row=current_row, column=2, value=f"{group_col.capitalize()}: {g_val}").font = Font(size=14, bold=True, color="1F4E79")
            ws.cell(row=current_row, column=2).fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
            current_row += 2
            
            for idx_m, m in enumerate(metrics):
                sub_df = df[(df[row_col] == r_val) & (df[group_col] == g_val)].copy()
                if sub_df.empty: 
                    continue
                    
                fig, ax = plt.subplots(figsize=(6, 4))
                
                for impl in sub_df['Implementation'].unique():
                    impl_df = sub_df[sub_df['Implementation'] == impl].copy()
                    
                    # Sort X-axis appropriately
                    if x_col == 'Workload':
                        impl_df['Workload'] = pd.Categorical(impl_df['Workload'], categories=workload_order, ordered=True)
                    impl_df = impl_df.sort_values(x_col)
                    
                    x = impl_df[x_col].astype(str)
                    y = impl_df[f'{m}_mean']
                    ci_l = impl_df[f'{m}_ci_lower']
                    ci_u = impl_df[f'{m}_ci_upper']
                    
                    yerr = [np.maximum(0, y - ci_l), np.maximum(0, ci_u - y)]
                    # Log scale handles lines better without markers
                    ax.errorbar(x, y, yerr=yerr, fmt='-', label=impl, capsize=5, capthick=1.5)
                    
                ax.set_title(m.replace('_', ' ').title(), fontsize=12)
                ax.set_xlabel(x_col.capitalize(), fontsize=10)
                ax.set_ylabel('Throughput', fontsize=10)
                ax.set_yscale('log')
                ax.legend(title="Implementation")
                ax.grid(True, linestyle='--', alpha=0.6)
                plt.tight_layout()
                
                # Format a safe filename
                temp_img_path = f"temp_{sheet_name.replace(' ', '')}_{r_val}_{g_val}_{m}.png".replace('/', '_')
                fig.savefig(temp_img_path, dpi=90)
                plt.close(fig)
                
                temp_images.append(temp_img_path)
                
                # Embed in Excel
                col_idx = 2 + (idx_m * 10) 
                ws.add_image(Image(temp_img_path), f"{get_column_letter(col_idx)}{current_row}")
                
            current_row += 22 
        current_row += 2

def create_experiment_dashboard(csv_path, output_excel_path):
    df = pd.read_csv(csv_path)
    metrics = [col.replace('_mean', '') for col in df.columns if col.endswith('_mean')]
    
    wb = Workbook()
    wb.remove(wb.active) # Remove the default empty sheet
    temp_images = []
    
    # --- Generate All 3 Tabs ---
    # 1. Original: X-axis = Workload, Rows = Size, Group = Density
    build_dashboard_sheet(wb, "By Size and Density", df, metrics, row_col='size', group_col='density', x_col='Workload', temp_images=temp_images)
    
    # 2. New Tab 1: X-axis = Size, Rows = Workload, Group = Density
    build_dashboard_sheet(wb, "By Workload and Density", df, metrics, row_col='Workload', group_col='density', x_col='size', temp_images=temp_images)
    
    # 3. New Tab 2: X-axis = Density, Rows = Workload, Group = Size
    build_dashboard_sheet(wb, "By Workload and Size", df, metrics, row_col='Workload', group_col='size', x_col='density', temp_images=temp_images)
    
    # Save & Cleanup
    wb.save(output_excel_path)
    print(f"Dashboard successfully saved to {output_excel_path}")
    
    for img_path in temp_images:
        if os.path.exists(img_path):
            os.remove(img_path)

if __name__ == "__main__":
    create_experiment_dashboard('results_oficial/confidence_interval.csv', 'results_oficial/Experiment_Dashboard.xlsx')

/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')
/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')
/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')
/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')
/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')
/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')
/tmp/ipykernel_2907/3012347547.py:71: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale('log')


Dashboard successfully saved to results_oficial/Experiment_Dashboard.xlsx


In [8]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load Data
df = pd.read_csv('results_oficial/confidence_interval.csv')
df = df[df['duration'] == 10000].copy()

# Map workloads
workload_mapping = {'read_only': 1, 'mostly_read': 2, 'mostly_write': 3}
df['Workload_num'] = df['Workload'].map(workload_mapping)
rev_workload_mapping = {1: 'Read Only', 2: 'Mostly Read', 3: 'Mostly Write'}

implementations = ['CSR', 'ADJACENCY_MATRIX', 'ADJACENCY_LIST']
impl_colors = {
    'CSR': 'rgba(31, 119, 180, 0.7)',             # Blue
    'ADJACENCY_MATRIX': 'rgba(214, 39, 40, 0.7)', # Red
    'ADJACENCY_LIST': 'rgba(44, 160, 44, 0.7)'    # Green
}

throughput_types = {
    'Total': 'throughput_total_mean',
    'Update': 'throughput_update_mean',
    'Read': 'throughput_read_mean'
}

# 2. Function to build a 1x3 Subplot with Log Scale
def build_combined_subplot_row(filtered_df, x_col, y_col, x_label, y_label):
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=[f"{tp_name} Throughput" for tp_name in throughput_types.keys()],
        horizontal_spacing=0.03
    )
    
    layout_updates = {}
    col_idx = 1
    
    for tp_name, tp_col in throughput_types.items():
        for impl in implementations:
            impl_df = filtered_df[filtered_df['Implementation'] == impl]
            if impl_df.empty: continue
            
            # PREVENT LOG(0) ERROR: Clip values so the absolute minimum is 1
            z_values = impl_df[tp_col].clip(lower=1)
            
            # Add 3D Surface (Manifold)
            if len(impl_df) >= 3:
                fig.add_trace(go.Mesh3d(
                    x=impl_df[x_col], y=impl_df[y_col], z=z_values,
                    opacity=0.6, color=impl_colors[impl], name=impl,
                    showlegend=(col_idx == 1) 
                ), row=1, col=col_idx)
                
            # Add Data Points
            fig.add_trace(go.Scatter3d(
                x=impl_df[x_col], y=impl_df[y_col], z=z_values,
                mode='markers',
                marker=dict(size=4, color=impl_colors[impl].replace('0.7', '1.0')),
                showlegend=False
            ), row=1, col=col_idx)
            
        scene_name = f'scene{col_idx if col_idx > 1 else ""}'
        if scene_name == 'scene': scene_name = 'scene1'
        
        xaxis_dict = dict(title=x_label)
        yaxis_dict = dict(title=y_label)
        
        if x_col == 'Workload_num': xaxis_dict.update(tickvals=[1,2,3], ticktext=['RO', 'MR', 'MW'])
        if y_col == 'Workload_num': yaxis_dict.update(tickvals=[1,2,3], ticktext=['RO', 'MR', 'MW'])
        
        # APPLY LOG SCALE TO Z-AXIS
        layout_updates[scene_name] = dict(
            xaxis=xaxis_dict, 
            yaxis=yaxis_dict, 
            zaxis=dict(title='Throughput (Log)', type='log') 
        )
        col_idx += 1
        
    fig.update_layout(**layout_updates)
    fig.update_layout(
        height=550, margin=dict(l=10, r=10, b=10, t=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
    )
    return fig.to_html(full_html=False, include_plotlyjs=False)

def get_html_header(title):
    return f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8"> <!-- THIS FIXES THE WEIRD CHARACTERS -->
        <title>{title}</title>
        <script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
        <style>
            body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #f8f9fa; margin: 0; padding: 20px; }}
            .header-box {{ background: #0d436b; color: white; padding: 15px; border-radius: 8px; margin: 40px 0 20px 0; text-align: center; }}
            .chart-container {{ background: white; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); padding: 10px; margin-bottom: 40px;}}
        </style>
    </head>
    <body>
    <h1 style="text-align: center; color: #333;">{title}</h1>
    """

# ==========================================
# FILE 1: X = Density, Y = Workload (Iterating over Sizes)
# ==========================================
html_1 = get_html_header("Dashboard 1: Fixado por Tamanho (X=Density, Y=Workload)")
for sz in sorted(df['size'].unique()):
    html_1 += f'<div class="header-box"><h2>Size = {sz}</h2></div><div class="chart-container">'
    html_1 += build_combined_subplot_row(df[df['size'] == sz], 'density', 'Workload_num', 'Density', 'Workload')
    html_1 += '</div>'
html_1 += "</body></html>"

with open("results_oficial/dashboard_1_size.html", "w", encoding="utf-8") as f:
    f.write(html_1)

# ==========================================
# FILE 2: X = Workload, Y = Size (Iterating over Densities)
# ==========================================
html_2 = get_html_header("Dashboard 2: Fixado por Densidade (X=Workload, Y=Size)")
for d in sorted(df['density'].unique()):
    html_2 += f'<div class="header-box"><h2>Density = {d}</h2></div><div class="chart-container">'
    html_2 += build_combined_subplot_row(df[df['density'] == d], 'Workload_num', 'size', 'Workload', 'Size')
    html_2 += '</div>'
html_2 += "</body></html>"

with open("results_oficial/dashboard_2_density.html", "w", encoding="utf-8") as f:
    f.write(html_2)

# ==========================================
# FILE 3: X = Size, Y = Density (Iterating over Workloads)
# ==========================================
html_3 = get_html_header("Dashboard 3: Fixado por Carga de Trabalho (X=Size, Y=Density)")
for w in sorted(df['Workload'].unique()):
    w_title = rev_workload_mapping[workload_mapping[w]]
    html_3 += f'<div class="header-box"><h2>Workload = {w_title}</h2></div><div class="chart-container">'
    html_3 += build_combined_subplot_row(df[df['Workload'] == w], 'size', 'density', 'Size', 'Density')
    html_3 += '</div>'
html_3 += "</body></html>"

with open("results_oficial/dashboard_3_workload.html", "w", encoding="utf-8") as f:
    f.write(html_3)

print("Success! Created 3 separate HTML files: 'dashboard_1_size.html', 'dashboard_2_density.html', and 'dashboard_3_workload.html'.")

Success! Created 3 separate HTML files: 'dashboard_1_size.html', 'dashboard_2_density.html', and 'dashboard_3_workload.html'.


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# 1. Load Data
df = pd.read_csv('ic_grafos.csv')
df = df[df['duration'] == 10000].copy()

# Map workloads to numbers for the X-axis
workload_mapping = {'read_only': 1, 'mostly_read': 2, 'mostly_write': 3}
df['Workload_num'] = df['Workload'].map(workload_mapping)

# Define Symbols and Offsets (Jitter) to prevent overlapping!
# CSR is shifted left (-0.15), Matrix stays center (0), List shifts right (+0.15)
impl_config = {
    'CSR': {'symbol': 'circle', 'offset': -0.15},
    'ADJACENCY_MATRIX': {'symbol': 'square', 'offset': 0.0},
    'ADJACENCY_LIST': {'symbol': 'diamond', 'offset': 0.15}
}

throughput_types = {
    'Total': 'throughput_total_mean',
    'Update': 'throughput_update_mean',
    'Read': 'throughput_read_mean'
}

# 2. Build the Single Combined 1x3 Dashboard
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=[f"{tp} Throughput" for tp in throughput_types.keys()],
    horizontal_spacing=0.02
)

layout_updates = {}
col_idx = 1

for tp_name, tp_col in throughput_types.items():
    
    # Get global min/max for this specific throughput type to lock the color scale safely
    valid_data = df[tp_col].clip(lower=1)
    cmin = np.log10(valid_data.min())
    cmax = np.log10(valid_data.max())
    
    show_colorbar = (col_idx == 3) # Only draw the colorbar on the last chart
    
    for impl, config in impl_config.items():
        plot_df = df[df['Implementation'] == impl].copy()
        
        if not plot_df.empty:
            log_throughput = np.log10(plot_df[tp_col].clip(lower=1))
            
            # APPLY JITTER: Offset the X coordinate slightly so they don't overlap
            jittered_x = plot_df['Workload_num'] + config['offset']
            
            fig.add_trace(go.Scatter3d(
                x=jittered_x,
                y=plot_df['size'],
                z=plot_df['density'],
                mode='markers',
                marker=dict(
                    symbol=config['symbol'],
                    size=10,               
                    color=log_throughput,  
                    colorscale='RdYlGn',   # Red -> Yellow -> Green
                    cmin=cmin,             
                    cmax=cmax,             
                    showscale=(show_colorbar and impl == 'ADJACENCY_LIST'), 
                    colorbar=dict(title="Throughput<br>(Log Scale)", x=1.02) if show_colorbar else None,
                    line=dict(width=1, color='black'), 
                    opacity=1.0 # Solid opacity since they no longer overlap
                ),
                name=impl,
                text=plot_df[tp_col], 
                hovertemplate=(
                    f"<b>{impl}</b><br>"
                    "<b>Workload:</b> %{{x}}<br>"
                    "<b>Size:</b> %{{y}}<br>"
                    "<b>Density:</b> %{{z}}<br>"
                    "<b>Throughput:</b> %{{text:,.1f}}<extra></extra>"
                )
            ), row=1, col=col_idx)

    # Format the 3D axes for this scene
    scene_name = f'scene{col_idx if col_idx > 1 else ""}'
    if scene_name == 'scene': scene_name = 'scene1'
    
    layout_updates[scene_name] = dict(
        xaxis=dict(
            title='Workload', 
            tickvals=[1,2,3], 
            ticktext=['RO', 'MR', 'MW'],
            range=[0.5, 3.5] # Add padding so the shifted icons don't hit the walls
        ),
        yaxis=dict(title='Size', type='category'), 
        zaxis=dict(title='Density'),
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.2))
    )
    col_idx += 1

# Apply layout updates
fig.update_layout(**layout_updates)
fig.update_layout(
    height=700,
    margin=dict(l=10, r=10, b=10, t=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
)

# 3. HTML Wrapper
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Combined Throughput Comparison</title>
    <script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #f8f9fa; margin: 0; padding: 20px; }}
        .header-box {{ background: #0d436b; color: white; padding: 15px; border-radius: 8px; margin: 10px 0 20px 0; text-align: center; }}
        .legend-bar {{ text-align: center; font-weight: bold; margin-bottom: 20px; padding: 15px; background: #e9ecef; border-radius: 8px; font-size: 15px; line-height: 1.6; }}
        .chart-container {{ background: white; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); padding: 10px; }}
    </style>
</head>
<body>
    <div class="header-box">
        <h1 style="margin:0;">Global Architecture Comparison</h1>
    </div>
    <div class="legend-bar">
        🟢 <b>Green</b> = High Throughput (Fast) &nbsp;&nbsp;|&nbsp;&nbsp; 🟡 <b>Yellow</b> = Medium &nbsp;&nbsp;|&nbsp;&nbsp; 🔴 <b>Red</b> = Low Throughput (Slow)<br>
        ● <b>Circle:</b> CSR &nbsp;&nbsp;|&nbsp;&nbsp; ■ <b>Square:</b> Adjacency Matrix &nbsp;&nbsp;|&nbsp;&nbsp; ◆ <b>Diamond:</b> Adjacency List<br>
        <span style="font-size:13px; font-weight:normal; color:#555;">(Icons are slightly shifted on the Workload axis to prevent overlapping)</span>
    </div>
    <div class="chart-container">
        {fig.to_html(full_html=False, include_plotlyjs=False)}
    </div>
</body>
</html>
"""

# Save the unified file
output_file = "dashboard_all_in_one.html"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Success! Generated '{output_file}'.")